## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

> [!NOTE]
> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/05-batch/setup/pyspark.md)

In [5]:
import pyspark

In [6]:
pyspark.__version__

'3.3.2'

## Question 2: Yellow October 2024

Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB

In [63]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-06 17:10:50--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
54.230.209.200, 54.230.209.126, 54.230.209.140, ...ychx.cloudfront.net)... 
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.200|:443... 
200 OKequest sent, awaiting response... 
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  59.7MB/s    in 1.0s    

2025-03-06 17:10:52 (59.7 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [64]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [196]:
df = spark.read.parquet('data/yellow_tripdata_2024-10.parquet')

In [ ]:
df = df.repartition(4)
df.write.parquet('data/yellow/2024/10/')

AnalysisException: path file:/home/dduke/GitHub/de-zoomcamp-hw/05-batch/code/data/yellow/2024/10 already exists.

In [66]:
!ls -l yellow/2024/10

total 99236
-rw-r--r-- 1 dduke dduke        0 Mar  6 17:11 _SUCCESS
-rw-r--r-- 1 dduke dduke 25406362 Mar  6 17:11 part-00000-74a4def3-7665-4cb7-a696-cb1051dcd8fd-c000.snappy.parquet
-rw-r--r-- 1 dduke dduke 25378641 Mar  6 17:11 part-00001-74a4def3-7665-4cb7-a696-cb1051dcd8fd-c000.snappy.parquet
-rw-r--r-- 1 dduke dduke 25430334 Mar  6 17:11 part-00002-74a4def3-7665-4cb7-a696-cb1051dcd8fd-c000.snappy.parquet
-rw-r--r-- 1 dduke dduke 25397583 Mar  6 17:11 part-00003-74a4def3-7665-4cb7-a696-cb1051dcd8fd-c000.snappy.parquet


## Question 3: Count records 

How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.

- 85,567
- 105,567
- 125,567
- 145,567

In [67]:
from pyspark.sql import functions as F

In [203]:
df_yellow = spark.read.parquet('data/yellow/2024/10/*')

In [100]:
df_yellow \
    .filter(F.to_date(df_yellow.tpep_pickup_datetime) == '2024-10-15') \
    .count()

# .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
# .withColumn('dropoff_date', F.to_date(df.tpep_dropoff_datetime)) \
# .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
# .filter(df.pickup_date == '2024-10-15') \

128909

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 122
- 142
- 162
- 182

In [168]:
def getHoursDiff(end_date, start_date):
    difference = end_date - start_date
    seconds = difference.total_seconds()
    hours = seconds/(60*60)
    return int(hours)

In [177]:
from pyspark.sql import types
from pyspark.sql import functions as F

hours_diff_udf = F.udf(getHoursDiff, returnType=types.IntegerType())

In [175]:
from pyspark.sql.functions import col
from pyspark.sql.functions import desc

df_yellow \
    .withColumn('duration', hours_diff_udf(df_yellow.tpep_dropoff_datetime, df_yellow.tpep_pickup_datetime)) \
    .select('duration') \
    .orderBy(col('duration').desc()) \
    .head(5)

[Row(duration=162),
 Row(duration=143),
 Row(duration=137),
 Row(duration=114),
 Row(duration=89)]

## Question 5: User Interface

Spark’s User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

In [178]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-06 18:21:20--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
54.230.209.72, 54.230.209.140, 54.230.209.200, ...rychx.cloudfront.net)... 
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.002s  

2025-03-06 18:21:21 (5.67 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [199]:
df_zones_lookup = spark.read.parquet('data/zones/*')
df_zones_lookup.registerTempTable('zones')

In [214]:
df_zones_lookup.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [204]:
df_yellow = spark.read.parquet('data/yellow/2024/10/*')
df_yellow.registerTempTable('yellow')

In [219]:
df_results = spark.sql("""
    SELECT
        z.Borough,
        z.Zone,
        COUNT(y.PULocationID)
    FROM yellow y
        LEFT OUTER JOIN zones z
            ON y.PULocationID = z.LocationID
    GROUP BY 1, 2
    ORDER BY 3 ASC
""")

In [220]:
df_results.show()

+-------------+--------------------+-------------------+
|      Borough|                Zone|count(PULocationID)|
+-------------+--------------------+-------------------+
|    Manhattan|Governor's Island...|                  1|
|Staten Island|       Arden Heights|                  2|
|        Bronx|       Rikers Island|                  2|
|       Queens|         Jamaica Bay|                  3|
|     Brooklyn| Green-Wood Cemetery|                  3|
|Staten Island|Eltingville/Annad...|                  4|
|Staten Island|Charleston/Totten...|                  4|
|Staten Island|       West Brighton|                  4|
|Staten Island|       Port Richmond|                  4|
|Staten Island|   Rossville/Woodrow|                  4|
|Staten Island|         Great Kills|                  6|
|        Bronx|        Crotona Park|                  6|
|Staten Island|Heartland Village...|                  7|
|Staten Island|     Mariners Harbor|                  7|
|Staten Island|Saint George/New